In [ ]:
#Data Cleaning
#The following preprocessing steps were performed:

#1. Loaded the raw air quality dataset.
#2. Audited data quality.
#3. Checked missing values.
#4. Removed duplicate records.
#5. Converted date column to datetime format.
#6. Created three new features:
   #- Pollution Range
   #- Update Hour
   #- Pollution Category
#7. Exported the cleaned dataset.

In [4]:
# Import Libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
# Display all columns
pd.set_option('display.max_columns', None)
# LOAD DATASET
df = pd.read_csv("../data/raw_dataset.csv")
# INITIAL DATA INSPECTION
print("\nDataset Shape:")
print(df.shape)
print("\nFirst 5 Rows:")
print(df.head())
print("\nColumn Names:")
print(df.columns.tolist())



Dataset Shape:
(4571, 11)

First 5 Rows:
  country           state        city  \
0   India  Andhra_Pradesh     Nellore   
1   India     West_Bengal      Howrah   
2   India   Uttar_Pradesh   Ghaziabad   
3   India  Andhra_Pradesh     Nellore   
4   India         Haryana  Mandikhera   

                                     station          last_update   latitude  \
0           Ambedkar Nagar, Nellore - APPCB   31-05-2026 10:00:00  14.438164   
1                   Dasnagar, Howrah - WBPCB  31-05-2026 10:00:00  22.602557   
2              Vasundhara, Ghaziabad - UPPCB  31-05-2026 10:00:00  28.660335   
3           Ambedkar Nagar, Nellore - APPCB   31-05-2026 10:00:00  14.438164   
4  General Hospital, Mandikhera(Nuh) - HSPCB  31-05-2026 10:00:00  27.900200   

   longitude pollutant_id  pollutant_min  pollutant_max  pollutant_avg  
0  79.966814        PM2.5           63.0          246.0          111.0  
1  88.310566        OZONE           15.0           93.0           26.0  
2  77.35725

In [3]:
#dataset information
print("\nDataset Information:")
print(df.info())


Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4571 entries, 0 to 4570
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   country        4571 non-null   object 
 1   state          4571 non-null   object 
 2   city           4571 non-null   object 
 3   station        4571 non-null   object 
 4   last_update    4571 non-null   object 
 5   latitude       4571 non-null   float64
 6   longitude      4571 non-null   float64
 7   pollutant_id   4571 non-null   object 
 8   pollutant_min  4204 non-null   float64
 9   pollutant_max  4204 non-null   float64
 10  pollutant_avg  4204 non-null   float64
dtypes: float64(5), object(6)
memory usage: 392.9+ KB
None


In [9]:
# DATA QUALITY AUDIT
print("DATA # Missing Values")
print("\nMissing Values:")
print(df.isnull().sum())

# Missing Percentage
print("\nMissing Value Percentage:")
missing_percentage = (df.isnull().sum() / len(df)) * 100
print(missing_percentage)

# Duplicate Records
print("\nDuplicate Records:")
duplicates = df.duplicated().sum()
print("Number of Duplicate Rows:", duplicates)

# Data Types
print("\nData Types:")
print(df.dtypes)

# Statistical Summary
print("\nStatistical Summary:")
print(df.describe())


DATA # Missing Values

Missing Values:
country            0
state              0
city               0
station            0
last_update        0
latitude           0
longitude          0
pollutant_id       0
pollutant_min    367
pollutant_max    367
pollutant_avg    367
dtype: int64

Missing Value Percentage:
country          0.000000
state            0.000000
city             0.000000
station          0.000000
last_update      0.000000
latitude         0.000000
longitude        0.000000
pollutant_id     0.000000
pollutant_min    8.028878
pollutant_max    8.028878
pollutant_avg    8.028878
dtype: float64

Duplicate Records:
Number of Duplicate Rows: 1263

Data Types:
country           object
state             object
city              object
station           object
last_update       object
latitude         float64
longitude        float64
pollutant_id      object
pollutant_min    float64
pollutant_max    float64
pollutant_avg    float64
dtype: object

Statistical Summary:
          lati

In [10]:
# BEFORE CLEANING
rows_before = df.shape[0]
columns_before = df.shape[1]

print("\nRows Before Cleaning:", rows_before)
print("Columns Before Cleaning:", columns_before)


Rows Before Cleaning: 4571
Columns Before Cleaning: 11


In [12]:
# CONVERT DATE COLUMN
df['last_update'] = pd.to_datetime(
    df['last_update'],
    dayfirst=True,
    errors='coerce'
)
print("\nDate Conversion Completed")


Date Conversion Completed


In [13]:
# HANDLE MISSING VALUES
print("\nMissing Values Before Treatment:")
print(df.isnull().sum())

# Forward Fill Missing Values
df.fillna(method='ffill', inplace=True)

# If any still remain, backward fill
df.fillna(method='bfill', inplace=True)

print("\nMissing Values After Treatment:")
print(df.isnull().sum())


Missing Values Before Treatment:
country            0
state              0
city               0
station            0
last_update        0
latitude           0
longitude          0
pollutant_id       0
pollutant_min    367
pollutant_max    367
pollutant_avg    367
dtype: int64

Missing Values After Treatment:
country          0
state            0
city             0
station          0
last_update      0
latitude         0
longitude        0
pollutant_id     0
pollutant_min    0
pollutant_max    0
pollutant_avg    0
dtype: int64


In [14]:
# REMOVE DUPLICATES
df.drop_duplicates(inplace=True)

rows_after_duplicates = df.shape[0]

print("\nDuplicates Removed:",
      rows_before - rows_after_duplicates)


Duplicates Removed: 1173


In [15]:
# FEATURE ENGINEERING
print("FEATURE ENGINEERING")

# Feature 1: Pollution Range
df['pollution_range'] = (
    df['pollutant_max']
    - df['pollutant_min']
)

# Feature 2: Update Hour
df['update_hour'] = (
    df['last_update']
    .dt.hour
)

# Feature 3: Pollution Category
def pollution_category(avg):

    if avg <= 50:
        return "Good"

    elif avg <= 100:
        return "Moderate"

    elif avg <= 200:
        return "Poor"

    else:
        return "Very Poor"

df['pollution_category'] = (
    df['pollutant_avg']
    .apply(pollution_category)
)

print("\nNew Features Created Successfully")


FEATURE ENGINEERING

New Features Created Successfully


In [17]:
# DATA VALIDATION
print("DATA VALIDATION")

print("\nFinal Dataset Shape:")
print(df.shape)

print("\nFinal Data Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

DATA VALIDATION

Final Dataset Shape:
(3398, 14)

Final Data Types:
country                       object
state                         object
city                          object
station                       object
last_update           datetime64[ns]
latitude                     float64
longitude                    float64
pollutant_id                  object
pollutant_min                float64
pollutant_max                float64
pollutant_avg                float64
pollution_range              float64
update_hour                    int32
pollution_category            object
dtype: object

Missing Values:
country               0
state                 0
city                  0
station               0
last_update           0
latitude              0
longitude             0
pollutant_id          0
pollutant_min         0
pollutant_max         0
pollutant_avg         0
pollution_range       0
update_hour           0
pollution_category    0
dtype: int64

Duplicate Rows:
0


In [18]:
# BEFORE VS AFTER COMPARISON
comparison = pd.DataFrame({
    'Before Cleaning': [rows_before, columns_before],
    'After Cleaning': [df.shape[0], df.shape[1]]
},
index=['Rows', 'Columns'])

print("\nBefore vs After Comparison:")
print(comparison)


Before vs After Comparison:
         Before Cleaning  After Cleaning
Rows                4571            3398
Columns               11              14


In [19]:
# FEATURE PREVIEW
print("\nNew Feature Preview:")
print(
    df[
        [
            'pollutant_min',
            'pollutant_max',
            'pollution_range',
            'update_hour',
            'pollution_category'
        ]
    ].head()
)



New Feature Preview:
   pollutant_min  pollutant_max  pollution_range  update_hour  \
0           63.0          246.0            183.0           10   
1           15.0           93.0             78.0           10   
2            4.0           28.0             24.0           10   
3           63.0          155.0             92.0           10   
4           25.0           27.0              2.0           10   

  pollution_category  
0               Poor  
1               Good  
2               Good  
3               Poor  
4               Good  


In [20]:
# FINAL DATASET PREVIEW
print("\nFinal Dataset Preview:")
print(df.head())

print("\nFinal Columns:")
print(df.columns.tolist())

# SAVE CLEANED DATASET
df.to_csv(
    "../data/cleaned_air_quality.csv",
    index=False
)


Final Dataset Preview:
  country           state        city  \
0   India  Andhra_Pradesh     Nellore   
1   India     West_Bengal      Howrah   
2   India   Uttar_Pradesh   Ghaziabad   
3   India  Andhra_Pradesh     Nellore   
4   India         Haryana  Mandikhera   

                                     station         last_update   latitude  \
0           Ambedkar Nagar, Nellore - APPCB  2026-05-31 10:00:00  14.438164   
1                   Dasnagar, Howrah - WBPCB 2026-05-31 10:00:00  22.602557   
2              Vasundhara, Ghaziabad - UPPCB 2026-05-31 10:00:00  28.660335   
3           Ambedkar Nagar, Nellore - APPCB  2026-05-31 10:00:00  14.438164   
4  General Hospital, Mandikhera(Nuh) - HSPCB 2026-05-31 10:00:00  27.900200   

   longitude pollutant_id  pollutant_min  pollutant_max  pollutant_avg  \
0  79.966814        PM2.5           63.0          246.0          111.0   
1  88.310566        OZONE           15.0           93.0           26.0   
2  77.357256          SO2       